In [1]:
import torch
import torch.nn as nn
import gensim
from datasets import load_dataset



In [2]:

ds = load_dataset("Ayon128/Banglish-English")
print(ds['train'][0])

README.md:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/1.20M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/479k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4000 [00:00<?, ? examples/s]

{'Banglish': 'Amar ei guitar ta cai.', 'English': 'I want this guitar.'}


In [18]:
train_data = ds['train']
# print(train_data[7000])
# d = []
# d.append([train_data['English']])
# print(d[0])
english_sentences_train = []
banglish_sentences_train = []


for item in train_data:
    english_sentences_train.append(item['English'])
    banglish_sentences_train.append(item['Banglish'])
english_sentences_train[:5]


['I want this guitar.',
 'There is a lot of work to be done in order to remove the conflicting position of BUET students and Chhatra League.',
 'The war had profound economic consequences.',
 'I agree.',
 'What they did was wrong.']

In [28]:
from collections import Counter
PAD_TOKEN = "<PAD>"
SOS_TOKEN = "<SOS>"
EOS_TOKEN = "<EOS>"
def preprocess_string(s):
    # Remove all non-word characters (everything except numbers and letters)
    s = re.sub(r"[^\w\s]", '', s)
    # Replace all runs of whitespaces with no space
    s = re.sub(r"\s+", '', s)
    # replace digits with no space
    s = re.sub(r"\d", '', s)
    return s

def preprocess_sentence(sentences, vocab):
    processed_sentences = []
    for s in sentences:
        indices = []
        indices.append(vocab.get('<SOS>', vocab['<UNK>']))
        for word in s.split():
             indices.append(vocab.get(word.lower(), vocab['<UNK>']))
        indices.append(vocab.get('<EOS>', vocab['<UNK>']))
        processed_sentences.append(indices)
            
            
        
        
        
        

def vocab_build(data):
    tokens = []
    tokens.append('<SOS>')
    tokens.append('<EOS>')
    tokens.append('<PAD>')
    okens.append('<UNK>')
    for sentence in data:
        for word in sentence.split():
            tokens.append(word.lower())

    word_freq = Counter(tokens)

    idx = 0
    vocab = {}
    for word, _ in word_freq.items():
        vocab[word] = idx
        idx += 1    
    return vocab
s = ['jhow are you', 'you are who']
word_freq = vocab_build(s)
ds = Counter(word_freq)
print(word_freq)
        

{'<SOS>': 0, '<EOS>': 1, '<PAD>': 2, 'jhow': 3, 'are': 4, 'you': 5, 'who': 6}


In [ ]:


class Encoder(nn.Module):
    def __init__(self, emb_size, hidden_dim, num_layers, vocab_size, batch_size):
        super(Encoder, self).__init__()
        self.emb_size = emb_size
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding (vocab_size, emb_size)
        self.lstm = nn.LSTM(input_size= emb_size,hidden_size = hidden_dim,num_layers=num_layers,batch_first=True)

    def forward(self, data):
        h_0 = torch.zeros(self.lstm.num_layers, batch_size, self.lstm.hidden_size).to(data.device)
        c_0 = torch.zeros(self.lstm.num_layers, batch_size, self.lstm.hidden_size).to(data.device)
        embedded_data = self.embedding(data)
        out, (hidden, cell) = self.lstm(embedded_data, (h_0, c_0))

        

In [ ]:
class Decoder(nn.Module):
    def __init__( self, emb_size,hidden_dim, num_layers, vocab_size):
        super(Decoder, self).__init__()
        self.emb_size = emb_size
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, emb_size)
        self.lstm = nn.LSTM(input_size=emb_size, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, data, hidden, cell):
        embedded_data = self.embedding(data) 
        out, (hidden, cell) = self.lstm(embedded_data, (hidden, cell))
        output = self.fc(out.squeeze(1))  # (batch_size, vocab_size)
        return output, hidden, cell


        


In [ ]:
import torch
import torch.nn as nn

# Defining LSTM
input_dim = 10  # Each word has a 10-dimensional embedding
hidden_dim = 20  # LSTM hidden size
num_layers = 2  # 2 LSTM layers
batch_size = 4  # 4 sequences per batch
seq_len = 5  # Each sequence has 5 words

lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True)

# Sample input tensor
input_tensor = torch.randn(batch_size, seq_len, input_dim)  # Shape: (4, 5, 10)

# Initial hidden & cell states
h_0 = torch.zeros(num_layers, batch_size, hidden_dim)  # Shape: (2, 4, 20)
c_0 = torch.zeros(num_layers, batch_size, hidden_dim)  # Shape: (2, 4, 20)

# Forward pass
out, (hidden, cell) = lstm(input_tensor, (h_0, c_0))

# Print shapes
print(f"Input shape: {input_tensor.shape}")  # (4, 5, 10)
print(f"Output shape: {out.shape}")  # (4, 5, 20)
print(f"Hidden state shape: {hidden.shape}")  # (2, 4, 20)
print(f"Cell state shape: {cell.shape}")  # (2, 4, 20)